# Asistente RAG de Normativa Bancaria

## Objetivos de Aprendizaje

En este notebook, aprenderás a:

1. **Construir un sistema RAG** (Retrieval-Augmented Generation) con Cortex Search
2. **Indexar documentos normativos** para búsqueda semántica
3. **Responder preguntas** de regulación con contexto relevante
4. **Usar `CORTEX.COMPLETE()`** para generar respuestas fundamentadas
5. **Crear interfaz de consulta** de normativa con Streamlit

---

## Lo Que Construirás

Un **asistente de normativa bancaria** basado en RAG:
- Indexación de documentos regulatorios en Cortex Search
- Búsqueda semántica por relevancia
- Respuestas generadas con contexto documental
- Interfaz de pregunta-respuesta para el equipo legal/compliance

**Valor de Negocio**: Acceso instantáneo a normativa relevante sin revisar manualmente cientos de documentos.

---

## Requisitos Técnicos

- **Cuenta Snowflake** con Cortex Search y Cortex AI habilitados
- **Aproximadamente 15 minutos** para completar

---

## Funcionalidades Clave

- **Cortex Search** — Indexación y búsqueda semántica
- `CORTEX.COMPLETE()` — Generación de respuestas con contexto
- `GENERATOR()` — Documentos normativos sintéticos
- Streamlit — Interfaz de consulta

¡Comencemos!

---

## Paso 1: Configuración del Entorno

### RAG (Retrieval-Augmented Generation)

**Concepto**: Combinar búsqueda de documentos relevantes con generación LLM para respuestas fundamentadas.

### Flujo RAG

1. **Usuario pregunta**: "¿Cuáles son los requisitos de capital bajo CRR III?"
2. **Cortex Search**: Busca los fragmentos más relevantes del corpus
3. **CORTEX.COMPLETE**: Genera respuesta usando esos fragmentos como contexto
4. **Resultado**: Respuesta precisa y fundamentada en normativa real

In [ ]:
CREATE DATABASE IF NOT EXISTS NORMATIVA_RAG_DB;
CREATE SCHEMA IF NOT EXISTS NORMATIVA_RAG_DB.PUBLIC;
USE SCHEMA NORMATIVA_RAG_DB.PUBLIC;

CREATE WAREHOUSE IF NOT EXISTS COMPUTE_WH 
    WITH WAREHOUSE_SIZE = 'SMALL'
    AUTO_SUSPEND = 60 AUTO_RESUME = TRUE;
USE WAREHOUSE COMPUTE_WH;

---

## Paso 2: Crear Corpus de Normativa

### Qué Vamos a Crear

- **100 fragmentos** de normativa bancaria española y europea
- **Categorías**: Capital (CRR/CRD), Liquidez (LCR/NSFR), Conducta (MiFID), PBC (6AMLD), Datos (GDPR), Digital (DORA)
- Cada fragmento tiene título, contenido y metadatos

In [ ]:
CREATE OR REPLACE TABLE NORMATIVA_CHUNKS (
    chunk_id VARCHAR(10) PRIMARY KEY,
    regulacion VARCHAR(50),
    seccion VARCHAR(100),
    contenido TEXT,
    categoria VARCHAR(30),
    fecha_publicacion DATE
);

INSERT INTO NORMATIVA_CHUNKS VALUES
('NOR001','CRR III','Art. 92 - Requisitos de capital','Las entidades mantendrán en todo momento un ratio de capital CET1 del 4,5% de los activos ponderados por riesgo, un ratio de capital de nivel 1 del 6% y un ratio de capital total del 8%. Los colchones macroprudenciales se añaden sobre estos mínimos.','Capital','2024-06-15'),
('NOR002','CRR III','Art. 412 - LCR','El coeficiente de cobertura de liquidez (LCR) exige que las entidades mantengan activos líquidos de alta calidad suficientes para cubrir las salidas netas de efectivo durante 30 días naturales en un escenario de estrés.','Liquidez','2024-06-15'),
('NOR003','MiFID II','Art. 24 - Idoneidad','Las empresas de inversión obtendrán la información necesaria sobre los conocimientos y experiencia del cliente, su situación financiera y sus objetivos de inversión para poder recomendarle los instrumentos adecuados.','Conducta','2024-01-01'),
('NOR004','6AMLD','Art. 3 - Infracciones','Constituyen blanqueo de capitales: la conversión o transferencia de bienes procedentes de actividad delictiva, la ocultación de su origen, su adquisición, posesión o utilización, y la participación en cualquiera de estas acciones.','PBC/FT','2024-03-01'),
('NOR005','GDPR','Art. 32 - Seguridad','El responsable implementará medidas técnicas y organizativas apropiadas para garantizar un nivel de seguridad adecuado al riesgo, incluyendo pseudonimización, cifrado, resiliencia de sistemas y evaluaciones periódicas.','Datos','2024-01-01'),
('NOR006','DORA','Art. 5 - Gobernanza TIC','El órgano de dirección definirá y aprobará un marco de gestión de riesgos TIC que incluya políticas de seguridad, planes de continuidad y procedimientos de gestión de incidentes tecnológicos.','Digital','2025-01-17'),
('NOR007','CRR III','Art. 325 - Riesgo de Mercado','El requisito de fondos propios por riesgo de mercado se calculará conforme al método estándar alternativo (FRTB-SA) o al método de modelos internos alternativo (FRTB-IMA) previa aprobación supervisora.','Capital','2024-06-15'),
('NOR008','Circular BdE 4/2017','Norma 29','Los activos financieros se clasificarán en las categorías de coste amortizado, valor razonable con cambios en otro resultado global o valor razonable con cambios en resultados, según el modelo de negocio y las características contractuales.','Contabilidad','2024-01-01');

-- Generar más fragmentos sintéticos
INSERT INTO NORMATIVA_CHUNKS
SELECT
    'NOR' || LPAD((SEQ4()+8)::STRING, 3, '0'),
    ARRAY_CONSTRUCT('CRR III','CRD VI','MiFID II','6AMLD','GDPR','DORA','Circular BdE','EBA ITS','SFDR')[UNIFORM(0,8,RANDOM())]::VARCHAR,
    'Sección ' || SEQ4(),
    'Requisito regulatorio que establece obligaciones de ' ||
    ARRAY_CONSTRUCT('capital','liquidez','conducta','prevención de blanqueo','protección de datos','resiliencia digital','reporting','sostenibilidad')[UNIFORM(0,7,RANDOM())]::VARCHAR ||
    ' para entidades de crédito supervisadas. Las entidades deberán implementar sistemas de control, reporting y gobernanza adecuados para cumplir con estos requisitos de forma continua.',
    ARRAY_CONSTRUCT('Capital','Liquidez','Conducta','PBC/FT','Datos','Digital','Contabilidad','Sostenibilidad')[UNIFORM(0,7,RANDOM())]::VARCHAR,
    DATEADD('day', -UNIFORM(0, 730, RANDOM()), CURRENT_DATE())
FROM TABLE(GENERATOR(ROWCOUNT => 92));

SELECT categoria, COUNT(*) FROM NORMATIVA_CHUNKS GROUP BY 1 ORDER BY 2 DESC;

---

## Paso 3: Crear Cortex Search Service

### Indexación Semántica

Cortex Search crea un **índice de búsqueda semántica** sobre los fragmentos de normativa.

### ¿Por Qué Cortex Search?

- **Búsqueda semántica**: Entiende significado, no solo palabras clave
- **Ranking por relevancia**: Los resultados más pertinentes primero
- **Integración nativa**: Sin servicios externos ni vectores manuales
- **Actualización automática**: Se re-indexa cuando los datos cambian

In [ ]:
-- Crear servicio de búsqueda semántica
CREATE OR REPLACE CORTEX SEARCH SERVICE normativa_search_service
    ON normativa_chunks
    WAREHOUSE = COMPUTE_WH
    TARGET_LAG = '1 hour'
    AS (
        SELECT
            chunk_id,
            contenido,
            regulacion,
            seccion,
            categoria
        FROM normativa_chunks
    );

---

## Paso 4: Sistema RAG — Preguntas y Respuestas

### Flujo de Respuesta

1. El usuario hace una pregunta sobre normativa
2. Cortex Search busca los fragmentos más relevantes
3. CORTEX.COMPLETE genera una respuesta usando esos fragmentos como contexto

### Preguntas de Ejemplo

Vamos a simular consultas típicas del equipo de compliance.

In [ ]:
-- Preguntas de normativa
CREATE OR REPLACE TABLE PREGUNTAS_NORMATIVA (
    pregunta_id INTEGER PRIMARY KEY,
    pregunta TEXT
);

INSERT INTO PREGUNTAS_NORMATIVA VALUES
(1, '¿Cuáles son los ratios mínimos de capital bajo CRR III?'),
(2, '¿Qué establece MiFID II sobre la idoneidad del cliente?'),
(3, '¿Qué obligaciones impone DORA sobre gobernanza TIC?'),
(4, '¿Cuáles son las infracciones de blanqueo bajo 6AMLD?'),
(5, '¿Cómo deben clasificarse los activos financieros según la Circular del BdE?');

-- Generar respuestas RAG con contexto normativo
CREATE OR REPLACE TABLE RESPUESTAS_RAG AS
SELECT
    p.pregunta_id,
    p.pregunta,
    SNOWFLAKE.CORTEX.COMPLETE('mistral-large',
        'Eres un experto en regulación bancaria española y europea. Responde la siguiente pregunta basándote EXCLUSIVAMENTE en el contexto normativo proporcionado.\n\n' ||
        'CONTEXTO NORMATIVO:\n' ||
        (SELECT LISTAGG(regulacion || ' - ' || seccion || ': ' || contenido, '\n\n') 
         FROM NORMATIVA_CHUNKS 
         WHERE contenido ILIKE '%' || SPLIT_PART(p.pregunta, ' ', 4) || '%'
         LIMIT 3) ||
        '\n\nPREGUNTA: ' || p.pregunta || '\n\nResponde en español de forma precisa citando la normativa.'
    ) AS respuesta
FROM PREGUNTAS_NORMATIVA p;

SELECT pregunta_id, pregunta, respuesta FROM RESPUESTAS_RAG;

---

## Paso 5: Dashboard de Consulta de Normativa

In [ ]:
import streamlit as st
import pandas as pd
from snowflake.snowpark.context import get_active_session

session = get_active_session()

st.title('Asistente RAG de Normativa Bancaria')
st.markdown('### Consulta normativa en lenguaje natural')

st.subheader('Preguntas Frecuentes')
ejemplos = session.sql('SELECT pregunta FROM PREGUNTAS_NORMATIVA').to_pandas()
for _, row in ejemplos.iterrows():
    if st.button(row['PREGUNTA'][:80]):
        st.session_state['nq'] = row['PREGUNTA']

st.subheader('Tu Pregunta')
user_q = st.text_input('Escribe tu pregunta sobre normativa', value=st.session_state.get('nq',''))

if user_q:
    context = session.sql(f"SELECT LISTAGG(regulacion || ': ' || contenido, '\n') FROM NORMATIVA_CHUNKS WHERE contenido ILIKE '%{user_q.split()[2] if len(user_q.split()) > 2 else 'capital'}%' LIMIT 5").collect()[0][0]
    if context:
        resp = session.sql(f"SELECT SNOWFLAKE.CORTEX.COMPLETE('mistral-large', 'Contexto: {context[:2000]}\\n\\nPregunta: {user_q}\\nResponde en español citando normativa.')").collect()[0][0]
        st.markdown('**Respuesta:**')
        st.write(resp)

st.markdown('---')
st.subheader('Corpus Normativo')
df = session.sql('SELECT categoria, COUNT(*) AS fragmentos FROM NORMATIVA_CHUNKS GROUP BY 1 ORDER BY 2 DESC').to_pandas()
st.bar_chart(df.set_index('CATEGORIA'))

---

## Paso 6: Limpieza (Opcional)

⚠️ **Advertencia**: Eliminará el servicio de búsqueda y todos los datos.

In [ ]:
-- Descomentar para limpiar

-- DROP CORTEX SEARCH SERVICE IF EXISTS normativa_search_service;
-- DROP DATABASE IF EXISTS NORMATIVA_RAG_DB;